<!-- NOTEBOOK_HEADER: This notebook is automatically generated by tools/build_notebook.py -->


# Kaggle training, verification, and evaluation

For the first run, execute the config, clone/install, CLI-contract, and verification cells manually and read the verification output before continuing.

`SESSION_PHASE` defaults to `setup_cache` so the first notebook pass always clones, verifies, restores HF state, and builds the cache; later sessions can switch to `train`, `eval`, or `all` when continuing work.

`train` runs all branch training; `eval` runs evaluation/results; `all` runs every phase. Every phase restores and syncs the same HF paths.


In [ ]:
# 1. Config
import os, re, sys, json, time, subprocess, threading, csv
from pathlib import Path

GITHUB_REPO_URL = "https://github.com/Algorithm-bot/ai-vs-real-face-detector.git"
GITHUB_BRANCH = "hybrid-detector-eval-overhaul"
KAGGLE_DATASET_MOUNT = Path("/kaggle/input/datasets/sahilsawantspit/ai-vs-real-face-detector-data")
SESSION_PHASE = "train"  # setup_cache | train | eval | all
VERIFY_STRICT = True
FORCE_RETRAIN = True
SEEDS = [42]
SUBSET_SEED = 42
MAX_TRAIN_IMAGES = 200_000
SEMANTIC_MODEL = "vit_large_patch14_clip_224.openai"
SEMANTIC_DIM = 768  # SemanticEncoder documents CLIP ViT-L/14's projected output width.
CACHE_BRANCHES = ["semantic", "physics", "freq"]
CACHE_BRANCHES_ARG = ",".join(CACHE_BRANCHES)
PREPROCESS = "crop"  # crop | resize | center_crop | no_resize | aspect_crop
CACHE_AUGMENT = True
# Augmentation parameters (match train.py defaults)
P_BLUR = 0.5
P_JPEG = 0.8
P_DOWNSCALE = 0.3
MAX_SIGMA = 3.0
MIN_QUALITY = 30
DOWNSCALE_RANGE = "0.5,0.9"
NUM_WORKERS = 10
SYNC_INTERVAL_S = 300
BRANCHES = ["freq_only"]
HPARAMS = {
    "semantic_only": dict(epochs=60, batch_size=128, lr=1e-3),
    "physics_only": dict(epochs=60, batch_size=128, lr=1e-3),
    "freq_only": dict(epochs=60, batch_size=128, lr=1e-3),
    "stage1": dict(epochs=4, batch_size=32, lr=1e-4),
    "full_hybrid": dict(epochs=4, batch_size=32, lr=1e-4),
}
HF_CHECKPOINT_REPO = "sahilSpit/checkpoints"
HF_CHECKPOINT_PRIVATE = True
WORKING = Path("/kaggle/working")
MODELS_ROOT = WORKING / "models_v2"
FEATURE_CACHE_DIR = WORKING / "feature_cache_v2"
EVAL_ROOT = WORKING / "evaluation_v2"
log_path = WORKING / "train.log"
assert SESSION_PHASE in {"all", "setup_cache", "train", "eval"}
assert not [mode for mode in BRANCHES if mode not in HPARAMS]
print(f"phase={SESSION_PHASE}; seeds={SEEDS}; cache branches={CACHE_BRANCHES_ARG}; preprocess={PREPROCESS}; augment={CACHE_AUGMENT}")

In [5]:
# 2. Clone and install (always run)
import shutil
clone_dir = WORKING / "ai-vs-real-face-detector"
if clone_dir.exists():
    shutil.rmtree(clone_dir)
subprocess.run(["git", "clone", "--depth", "1", "--branch", GITHUB_BRANCH, GITHUB_REPO_URL, str(clone_dir)], check=True)

candidates = [clone_dir, clone_dir / "ai-vs-real-face-detector"]
project_root = next((p for p in candidates if (p / "requirements.txt").exists()), None)
assert project_root is not None, f"requirements.txt not found in {[str(c) for c in candidates]}"

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(project_root / "requirements.txt")], check=True)
sys.path.insert(0, str(project_root))
print("Project root:", project_root)

Cloning into '/kaggle/working/ai-vs-real-face-detector'...


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.5/44.5 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.0/63.0 MB 29.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.0/75.0 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 56.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.8/80.8 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 244.0/244.0 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 MB 38.0 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.48.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
kaggle-environments 1.27.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
mne 1.13.2 requires numpy<3,>=2.1, but you have numpy 1.26.4 which is incompatible.
bayesian-optimization 3.3.0 requires numpy>=2.1.3; python_full_version >= "3.13", but you have numpy 1.26.4 which is incompatible.
cesium 0.12.4 requires numpy<3.0,>=2.0, but you have numpy 1.26.4 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.20.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 2.3.3 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
dopamine-rl 4.1.2 requires g

Project root: /kaggle/working/ai-vs-real-face-detector/ai-vs-real-face-detector


In [6]:
# Run once per session, right after the clone/install cell and BEFORE the train cell.
# Idempotent: already-patched files are left alone. `project_root` comes from the clone cell.
def _patch(rel, edits):
    p = project_root / rel
    s = p.read_text(encoding="utf-8")
    for tag, old, new in edits:
        if new in s:
            print(f"{rel}: {tag}: already patched"); continue
        assert s.count(old) == 1, f"{rel}: {tag}: expected text not found exactly once"
        s = s.replace(old, new, 1)
        print(f"{rel}: {tag}: patched")
    p.write_text(s, encoding="utf-8")

# 1) cache check: "deep" is the end-to-end CNN branch and is never stored in the feature cache
_patch("src/eval/cache_meta.py", [(
    "ignore 'deep' in branch comparison",
    "return normalise_branches(cached) == normalise_branches(requested)",
    '# "deep" is the end-to-end CNN branch; it is never stored in the feature cache.\n'
    '        return normalise_branches(cached) == [b for b in normalise_branches(requested) if b != "deep"]',
)])

# 2) full-hybrid training/eval loop: PRNU is optional (off by default); pass the freq features to the model
_patch("src/train.py", [
    ("optional prnu + freq batch keys",
     '                prnu = batch["prnu"].to(\n                    device,\n                    non_blocking=True,\n                )\n',
     '                # prnu is off by default (branches: deep,physics,semantic,freq);\n'
     '                # only move it to the device when the dataset provides it.\n'
     '                prnu = (\n                    batch["prnu"].to(device, non_blocking=True)\n                    if "prnu" in batch else None\n                )\n'
     '                freq = (\n                    batch["freq"].to(device, non_blocking=True)\n                    if "freq" in batch else None\n                )\n'),
    ("pass freq_features to the model",
     '                logits, _ = model(\n                    images,\n                    physics,\n                    prnu,\n                    semantic,\n                )\n',
     '                logits, _ = model(\n                    images,\n                    physics,\n                    prnu,\n                    semantic,\n                    freq_features=freq,\n                )\n'),
])

# 3) evaluation: do not compute PRNU for a full_hybrid checkpoint that was trained without the prnu branch
_patch("src/evaluate.py", [(
    "no live PRNU for non-PRNU hybrids",
    '"use_prnu": mode in {"full_hybrid", "prnu_only"} or "prnu" in str(checkpoint_args.get("branches") or ""),',
    '"use_prnu": mode == "prnu_only" or "prnu" in str(checkpoint_args.get("branches") or "")'
    ' or (mode == "full_hybrid" and not checkpoint_args.get("branches")),',
)])

src/eval/cache_meta.py: ignore 'deep' in branch comparison: patched
src/train.py: optional prnu + freq batch keys: already patched
src/train.py: pass freq_features to the model: already patched
src/evaluate.py: no live PRNU for non-PRNU hybrids: patched


In [ ]:
# 2a. Assert every notebook flag exists in the real argparse help.
def has_flag(help_text, flag):
    return re.search(rf"(?<![A-Za-z0-9_-]){re.escape(flag)}(?![A-Za-z0-9_-])", help_text) is not None

help_contract = {
    "src/cache_features.py": [
        "--data-dir", "--cache-dir", "--workers", "--semantic-model", "--preprocess",
        "--augment", "--batch-size", "--max-train-images", "--branches", "--seed",
        "--subset-seed", "--p-blur", "--p-jpeg", "--p-downscale", "--max-sigma",
        "--min-quality", "--downscale-range"
    ],
    "src/train.py": [
        "--mode", "--data-dir", "--output-dir", "--epochs", "--batch-size", "--lr",
        "--num-workers", "--max-train-images", "--seed", "--subset-seed",
        "--feature-cache", "--branches", "--branch-dropout", "--fusion-mode",
        "--semantic-model", "--semantic-dim", "--preprocess", "--p-blur",
        "--p-jpeg", "--p-downscale", "--max-sigma", "--min-quality",
        "--downscale-range"
    ],
    "src/train_cached.py": [
        "--mode", "--data-dir", "--output-dir", "--epochs", "--batch-size", "--lr",
        "--feature-cache", "--max-train-images", "--seed", "--subset-seed",
        "--preprocess", "--p-blur", "--p-jpeg", "--p-downscale", "--max-sigma",
        "--min-quality", "--downscale-range"
    ],
    "src/evaluate.py": [
        "--checkpoint", "--data-dir", "--output-dir", "--feature-cache",
        "--splits-yaml", "--eval-set", "--n-bootstrap", "--seed"
    ],
    "src/calibrate_threshold.py": [
        "--checkpoint", "--data-dir", "--feature-cache", "--splits-yaml"
    ],
    "src/stack_fusion.py": [
        "--predictions", "--splits-yaml", "--output-dir", "--n-bootstrap", "--seed"
    ],
    "src/diagnose_branches.py": [
        "--feature-cache", "--predictions", "--splits-yaml", "--output"
    ],
    "kaggle_verify.py": [
        "--data-dir", "--feature-cache"
    ],
}
from concurrent.futures import ThreadPoolExecutor
def _read_help(item):
    relative, flags = item
    env = {**os.environ, "BRANCH_PROJECT_ROOT": str(project_root), "BRANCH_FEATURE_CACHE": str(FEATURE_CACHE_DIR)}
    result = subprocess.run([sys.executable, str(project_root / relative), "--help"], text=True, capture_output=True, env=env)
    text = result.stdout + result.stderr
    missing = [flag for flag in flags if not has_flag(text, flag)]
    return relative, {"returncode": result.returncode, "missing": missing}
with ThreadPoolExecutor(max_workers=len(help_contract)) as pool:
    help_results = dict(pool.map(_read_help, help_contract.items()))
for relative, result in help_results.items():
    print(relative, result)
assert all(row["returncode"] == 0 and not row["missing"] for row in help_results.values()), help_results
print("CLI help contract: OK")

In [8]:
# 2b. Verification deferred until cache construction completes.
if SESSION_PHASE in {"all", "setup_cache"}:
    print("Verification deferred until after cache construction.")
else:
    print(f"Skipping verification for phase {SESSION_PHASE}")

Skipping verification for phase train


In [9]:
# 3. Restore and sync HF artifacts (all phases)
from huggingface_hub import HfApi, snapshot_download
from kaggle_secrets import UserSecretsClient
hf_token = UserSecretsClient().get_secret("HF_TOKEN")
api = HfApi(token=hf_token)
os.environ["HF_TOKEN"] = hf_token
api.create_repo(repo_id=HF_CHECKPOINT_REPO, repo_type="model", private=HF_CHECKPOINT_PRIVATE, exist_ok=True)
allow_patterns = ["feature_cache_v2/*", "evaluation_v2/*", "logs/*"] + [f"models_v2/{mode}/seed_{seed}/*" for mode in BRANCHES for seed in SEEDS]
# Attempt to restore prior artifacts from HF
try:
    snapshot_download(repo_id=HF_CHECKPOINT_REPO, repo_type="model", token=hf_token, local_dir=str(WORKING), allow_patterns=allow_patterns)
except Exception as exc:
    print("No prior artifacts restored:", exc)
SYNC_DIRS = {FEATURE_CACHE_DIR: "feature_cache_v2", EVAL_ROOT: "evaluation_v2"}
for mode in BRANCHES:
    for seed in SEEDS:
        SYNC_DIRS[MODELS_ROOT / mode / f"seed_{seed}"] = f"models_v2/{mode}/seed_{seed}"
uploaded = {}

def sync_once(min_age=60):
    for local_dir, remote_dir in SYNC_DIRS.items():
        if not local_dir.exists():
            continue
        for path in local_dir.rglob("*"):
            if not path.is_file():
                continue
            st = path.stat()
            if time.time() - st.st_mtime < min_age:
                continue
            if uploaded.get(str(path)) == st.st_mtime:
                continue
            try:
                api.upload_file(
                    path_or_fileobj=str(path),
                    path_in_repo=f"{remote_dir}/{path.relative_to(local_dir).as_posix()}",
                    repo_id=HF_CHECKPOINT_REPO,
                    repo_type="model",
                )
                uploaded[str(path)] = st.st_mtime
            except Exception as exc:
                print(f"WARN: sync failed for {path}: {exc}")

stop_sync = threading.Event()
def _sync_loop():
    while not stop_sync.is_set():
        sync_once()
        stop_sync.wait(SYNC_INTERVAL_S)
threading.Thread(target=_sync_loop, daemon=True).start()


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 28 files:   0%|          | 0/28 [00:00<?, ?it/s]

In [ ]:
# 4. Build/reuse cache (setup_cache only)
if SESSION_PHASE in {"all", "setup_cache"}:
    cache_cmd = [
        sys.executable,
        str(project_root / "src/cache_features.py"),
        "--data-dir", str(KAGGLE_DATASET_MOUNT),
        "--cache-dir", str(FEATURE_CACHE_DIR),
        "--workers", str(NUM_WORKERS),
        "--semantic-model", SEMANTIC_MODEL,
        "--preprocess", PREPROCESS,
        "--batch-size", "64",
        "--max-train-images", str(MAX_TRAIN_IMAGES),
        "--branches", CACHE_BRANCHES_ARG,
        "--seed", str(SEEDS[0]),
        "--subset-seed", str(SUBSET_SEED),
        "--p-blur", str(P_BLUR),
        "--p-jpeg", str(P_JPEG),
        "--p-downscale", str(P_DOWNSCALE),
        "--max-sigma", str(MAX_SIGMA),
        "--min-quality", str(MIN_QUALITY),
        "--downscale-range", DOWNSCALE_RANGE,
    ]
    if CACHE_AUGMENT:
        cache_cmd.append("--augment")
    assert subprocess.run(cache_cmd, env={**os.environ, "HF_TOKEN": hf_token, "PYTHONUNBUFFERED": "1"}).returncode == 0
    from src.eval.cache_meta import load_cache_meta, normalise_branches
    for split in ("train", "val", "test"):
        assert (FEATURE_CACHE_DIR / f"{split}.npz").exists(), f"missing {split}.npz"
    meta = load_cache_meta(FEATURE_CACHE_DIR)
    assert normalise_branches(meta.get("branches")) == sorted(CACHE_BRANCHES)
    assert meta.get("max_train_images") == MAX_TRAIN_IMAGES
    assert meta.get("semantic_model") == SEMANTIC_MODEL
    assert meta.get("semantic_dim") == SEMANTIC_DIM
    assert meta.get("preprocess") == PREPROCESS
    assert bool(meta.get("augment")) is CACHE_AUGMENT
    assert abs(meta.get("p_blur", 0) - P_BLUR) < 1e-6, f"p_blur mismatch: {meta.get('p_blur')} vs {P_BLUR}"
    assert abs(meta.get("p_jpeg", 0) - P_JPEG) < 1e-6, f"p_jpeg mismatch: {meta.get('p_jpeg')} vs {P_JPEG}"
    assert abs(meta.get("p_downscale", 0) - P_DOWNSCALE) < 1e-6, f"p_downscale mismatch: {meta.get('p_downscale')} vs {P_DOWNSCALE}"
    assert abs(meta.get("max_sigma", 0) - MAX_SIGMA) < 1e-6, f"max_sigma mismatch: {meta.get('max_sigma')} vs {MAX_SIGMA}"
    assert meta.get("min_quality") == MIN_QUALITY, f"min_quality mismatch: {meta.get('min_quality')} vs {MIN_QUALITY}"
    assert abs(meta.get("downscale_min", 0) - 0.5) < 1e-6, f"downscale_min mismatch: {meta.get('downscale_min')} vs 0.5"
    assert abs(meta.get("downscale_max", 0) - 0.9) < 1e-6, f"downscale_max mismatch: {meta.get('downscale_max')} vs 0.9"
    print("cache metadata contract: OK")

In [ ]:
# 5. Train every mode and seed (train only). Stage1 deliberately has no --feature-cache.
def run_and_log(cmd, extra_env=None):
    env = {**os.environ, "PYTHONUNBUFFERED": "1", "HF_TOKEN": hf_token, **(extra_env or {})}
    state = {"last_bar": 0.0}
    def emit(raw, log):
        line = raw.decode("utf-8", errors="replace").rstrip()
        if not line:
            return
        bar = "it/s]" in line or "s/it]" in line or "%|" in line
        if bar and "100%|" not in line:
            if time.time() - state["last_bar"] < 30:
                return
            state["last_bar"] = time.time()
        print(line, flush=True)
        log.write(line)
        log.write(chr(10))
        log.flush()
    with open(log_path, "a", encoding="utf-8") as log:
        proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, env=env)
        buffer = b""
        while chunk := proc.stdout.read1(4096):
            buffer += chunk
            parts = re.split(rb"\r\n|\r|\n", buffer)
            buffer = parts.pop()
            for part in parts:
                emit(part, log)
        if buffer:
            emit(buffer, log)
        proc.wait()
    return proc.returncode

def build_cmd(mode, seed):
    hp, out = HPARAMS[mode], MODELS_ROOT / mode / f"seed_{seed}"
    common = [
        "--mode", mode,
        "--data-dir", str(KAGGLE_DATASET_MOUNT),
        "--output-dir", str(out),
        "--epochs", str(hp["epochs"]),
        "--batch-size", str(hp["batch_size"]),
        "--lr", str(hp["lr"]),
        "--seed", str(seed),
        "--max-train-images", str(MAX_TRAIN_IMAGES),
        "--subset-seed", str(SUBSET_SEED),
        "--preprocess", PREPROCESS,
        "--p-blur", str(P_BLUR),
        "--p-jpeg", str(P_JPEG),
        "--p-downscale", str(P_DOWNSCALE),
        "--max-sigma", str(MAX_SIGMA),
        "--min-quality", str(MIN_QUALITY),
        "--downscale-range", DOWNSCALE_RANGE,
    ]
    if mode in {"semantic_only", "physics_only", "freq_only"}:
        return [
            sys.executable, "-u", str(project_root / "src/train_cached.py"),
            *common,
            "--feature-cache", str(FEATURE_CACHE_DIR),
            "--semantic-model", SEMANTIC_MODEL,
            "--semantic-dim", str(SEMANTIC_DIM),
            "--branches", CACHE_BRANCHES_ARG,
        ], {"BRANCH_PROJECT_ROOT": str(project_root), "BRANCH_FEATURE_CACHE": str(FEATURE_CACHE_DIR)}
    cmd = [
        sys.executable, "-u", str(project_root / "src/train.py"),
        *common,
        "--data-source", "local",
        "--num-workers", str(NUM_WORKERS),
    ]
    if mode == "full_hybrid":
        cmd += [
            "--feature-cache", str(FEATURE_CACHE_DIR),
            "--semantic-model", SEMANTIC_MODEL,
            "--semantic-dim", str(SEMANTIC_DIM),
            "--branches", "deep,physics,semantic,freq",
            "--branch-dropout", "0.15",
            "--fusion-mode", "gated",
        ]
    return cmd, {}

# Periodic log upload thread
def _log_loop():
    while not stop_sync.is_set():
        if log_path.exists():
            try:
                api.upload_file(path_or_fileobj=str(log_path), path_in_repo="logs/train.log", repo_id=HF_CHECKPOINT_REPO, repo_type="model")
            except Exception as exc:
                print("WARN: log sync failed:", exc)
        stop_sync.wait(120)  # Every 2 minutes
threading.Thread(target=_log_loop, daemon=True).start()

if SESSION_PHASE in {"all", "train"}:
    skipped = []
    for seed in SEEDS:
        for mode in BRANCHES:
            out = MODELS_ROOT / mode / f"seed_{seed}"
            if (out / f"{mode}_history.json").exists() and not FORCE_RETRAIN:
                skipped.append(f"{mode}/seed_{seed}")
                continue
            cmd, env = build_cmd(mode, seed)
            assert ("--feature-cache" in cmd) == (mode != "stage1"), cmd
            rc = run_and_log(cmd, env); sync_once(min_age=0)
            if rc:
                raise RuntimeError(f"{mode}/seed_{seed} failed: {rc}")
    print("Skipped (already finished):", skipped or "none")
else:
    print(f"Skipping training for phase {SESSION_PHASE}")

In [12]:
# 6. Evaluate, calibrate, stack, and diagnose (eval only); fail on any subprocess error.
if SESSION_PHASE in {"all", "eval"}:
    import pandas as pd
    from src.eval.splits import load_split_spec
    from src.eval.tables import build_eval_tables
    failures, evaluation_rows = {}, []
    for seed in SEEDS:
        for mode in BRANCHES:
            root = MODELS_ROOT / mode / f"seed_{seed}"; checkpoint = root / f"{mode}_best.pt"; output = EVAL_ROOT / mode / f"seed_{seed}"
            eval_cmd = [sys.executable, str(project_root / "src/evaluate.py"), "--checkpoint", str(checkpoint), "--data-dir", str(KAGGLE_DATASET_MOUNT), "--output-dir", str(output), "--feature-cache", str(FEATURE_CACHE_DIR), "--splits-yaml", str(project_root / "eval/splits.yaml"), "--eval-set", "all", "--n-bootstrap", "1000", "--seed", str(seed)]
            cal_cmd = [sys.executable, str(project_root / "src/calibrate_threshold.py"), "--checkpoint", str(checkpoint), "--data-dir", str(KAGGLE_DATASET_MOUNT), "--feature-cache", str(FEATURE_CACHE_DIR), "--splits-yaml", str(project_root / "eval/splits.yaml")]
            evaluation_rows.append({"mode": mode, "seed": seed, "eval ok": run_and_log(eval_cmd) == 0, "calibrate ok": run_and_log(cal_cmd) == 0, "stack ok": None, "diagnose ok": None})
    by_seed = {seed: [] for seed in SEEDS}
    for row in evaluation_rows:
        prediction = EVAL_ROOT / row["mode"] / f"seed_{row['seed']}" / "predictions.csv"
        if prediction.exists(): by_seed[row["seed"]].append(prediction)
    for seed, paths in by_seed.items():
        merged = None
        for path in paths:
            df = pd.read_csv(path); merged = df if merged is None else merged.merge(df.drop(columns=[c for c in ["folder", "source", "label", "p_ai", "logit_real", "logit_ai", "split"] if c in df]), on="path", how="outer")
        combined = EVAL_ROOT / f"combined_seed_{seed}.csv"; merged.to_csv(combined, index=False)
        stack_cmd = [sys.executable, str(project_root / "src/stack_fusion.py"), "--predictions", str(combined), "--splits-yaml", str(project_root / "eval/splits.yaml"), "--output-dir", str(EVAL_ROOT / "stacking" / f"seed_{seed}"), "--seed", str(seed)]
        diag_cmd = [sys.executable, str(project_root / "src/diagnose_branches.py"), "--predictions", str(combined), "--splits-yaml", str(project_root / "eval/splits.yaml"), "--output", str(EVAL_ROOT / f"diagnostics_seed_{seed}.csv")]
        stack_ok, diag_ok = run_and_log(stack_cmd) == 0, run_and_log(diag_cmd) == 0
        for row in evaluation_rows:
            if row["seed"] == seed: row["stack ok"], row["diagnose ok"] = stack_ok, diag_ok
    summary = pd.DataFrame(evaluation_rows); print(summary.to_string(index=False))

    # Write summary table to evaluation_v2/ as JSON and CSV
    summary_csv_path = EVAL_ROOT / "summary.csv"
    summary_json_path = EVAL_ROOT / "summary.json"
    summary.to_csv(summary_csv_path, index=False)
    summary.to_json(summary_json_path, indent=2)
    print(f"Summary table written to {summary_csv_path} and {summary_json_path}")

    if not summary[["eval ok", "calibrate ok", "stack ok", "diagnose ok"]].fillna(False).all().all(): raise RuntimeError("Evaluation subprocess failure(s); see summary above.")
else:
    print(f"Skipping evaluation for phase {SESSION_PHASE}")


Skipping evaluation for phase train


In [13]:
# 7. Per-seed fold table and multi-seed mean +/- std.
if SESSION_PHASE in {"all", "eval"}:
    from src.eval.splits import load_split_spec
    from src.eval.tables import build_eval_tables
    spec, rows = load_split_spec(project_root / "eval/splits.yaml"), []
    for seed in SEEDS:
        df = pd.read_csv(EVAL_ROOT / f"combined_seed_{seed}.csv")
        for name, folders in {"fold_a": spec.dev, "fold_b": spec.report, "progan": spec.progan}.items():
            part = df[df.folder.map(lambda f: spec.assign(str(f)) == ({"fold_a":"dev", "fold_b":"report", "progan":"progan"}[name]))]
            tables = build_eval_tables(part.to_dict("records"), n_bootstrap=1000, seed=seed)
            rows.append({"seed": seed, "split": name, "metrics": tables["at_0.5"]["overall"]})
    comparison = {r["split"]: {} for r in rows}
    for r in rows: comparison[r["split"]][f"seed_{r['seed']}"] = r["metrics"]
    required = {"fold_a", "fold_b", "progan"}; found = set(comparison)
    print("Found result keys:", sorted(found)); assert required <= found, f"Missing result keys: {required - found}"
    (EVAL_ROOT / "branch_comparison_v2.json").write_text(json.dumps(comparison, indent=2, default=str))
    for split in sorted(required):
        aucs = [comparison[split][f"seed_{seed}"].get("roc_auc") for seed in SEEDS]
        print(f"{split}: AUC mean +/- std = {sum(aucs)/len(aucs):.4f} +/- {pd.Series(aucs).std(ddof=0):.4f}")
    if len(SEEDS) == 1: print("One seed: bootstrap CIs are image-level resampling only, not training variance.")


# Notes

Current branches: `semantic_only`, `physics_only`, `freq_only`, `stage1`, `full_hybrid`. Checkpoints are at `models_v2/<mode>/seed_<seed>/`; the comparison JSON is `evaluation_v2/branch_comparison_v2.json`.

To retrain one branch, set `BRANCHES = ["freq_only"]` (or another current mode) and run the train phase. Pull a checkpoint with `hf_hub_download(repo_id=HF_CHECKPOINT_REPO, filename="models_v2/freq_only/seed_42/freq_only_best.pt")".


In [14]:
# 8. Final sync (every phase)
stop_sync.set()
for attempt in range(3):
    sync_once(min_age=0)
    remote = set(api.list_repo_files(repo_id=HF_CHECKPOINT_REPO, repo_type="model"))
    missing = [f"{rd}/{p.relative_to(ld).as_posix()}" for ld, rd in SYNC_DIRS.items() if ld.exists()
               for p in ld.rglob("*") if p.is_file() and f"{rd}/{p.relative_to(ld).as_posix()}" not in remote]
    if not missing:
        print("All synced files verified on HF."); break
    print(f"attempt {attempt+1}/3: {len(missing)} missing, e.g. {missing[:3]}"); time.sleep(10)
else:
    print(f"WARN: still missing on HF: {missing[:10]}")
for attempt in range(3):
    try:
        if log_path.exists():
            api.upload_file(path_or_fileobj=str(log_path), path_in_repo="logs/train.log", repo_id=HF_CHECKPOINT_REPO, repo_type="model")
        break
    except Exception as exc:
        print("train.log upload failed:", exc); time.sleep(5)
print(f"https://huggingface.co/{HF_CHECKPOINT_REPO}")


All synced files verified on HF.
https://huggingface.co/sahilSpit/checkpoints
